In [3]:
import os
import numpy as np
import pandas as pd

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260615_154932"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
trades = pd.read_parquet(os.path.join(folder_path, "trades.parquet"))
quotes = pd.read_parquet(os.path.join(folder_path, "quotes.parquet"))
fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))
events = pd.read_parquet(os.path.join(folder_path, "events.parquet"))

def add_trade_impact_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Adds future return at multiple time horizons using snapshots.
    Assumes both inputs are already sorted by ts.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        out[f"trade_return_{h}ms"] = (
            (merged[f"future_mid_{h}"] - out["price"]) / out["price"]
        )

    return out

def label_toxicity_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Toxicity = price moved against aggressor after trade.
    Multi-horizon version.
    Assumes sorted inputs.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        move = merged[f"future_mid_{h}"] - out["price"]

        out[f"toxicity_{h}ms"] = np.where(
            out["side"] == "BUY",
            move < 0,
            move > 0
        ).astype(int)

        out[f"signed_impact_{h}ms"] = np.where(
            out["side"] == "BUY",
            -move,
            move
        )

        out[f"signed_impact_bps_{h}ms"] = (
            10000 * out[f"signed_impact_{h}ms"] / out["price"]
        )

    return out

def align_trades_to_snapshots(trades: pd.DataFrame, snapshots: pd.DataFrame):

    snap = snapshots[["ts", "mid", "best_bid", "best_ask"]].rename(columns={
        "mid": "snap_mid",
        "best_bid": "snap_bid",
        "best_ask": "snap_ask"
    })

    merged = pd.merge_asof(trades, snap, on="ts", direction="backward")

    trades = merged.copy()

    trades["snapshot_mid"] = trades["snap_mid"]
    trades["snapshot_bid"] = trades["snap_bid"]
    trades["snapshot_ask"] = trades["snap_ask"]
    trades["spread"] = trades["snap_ask"] - trades["snap_bid"]

    return trades

def label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000)):

    snap_ts = snapshots["ts"].values
    snap_mid = snapshots["mid"].values

    fills = fills.copy()

    base_idx = np.searchsorted(snap_ts, fills["ts"].values, side="right") - 1
    base_idx = np.clip(base_idx, 0, len(snap_ts) - 1)

    fills["snap_mid"] = snap_mid[base_idx]

    fill_ts = fills["ts"].values
    fill_price = fills["price"].values

    for h in horizons_ms:
        target = fill_ts + h

        idx = np.searchsorted(snap_ts, target, side="left")
        idx = np.clip(idx, 0, len(snap_ts) - 1)

        fills[f"markout_{h}ms"] = snap_mid[idx] - fill_price
        fills[f"adverse_{h}ms"] = fill_price - snap_mid[idx]

    return fills

def forward_index(ts, horizon):
    n = len(ts)
    j = 0
    idx = np.empty(n, dtype=int)

    for i in range(n):
        while j < n and ts[j] < ts[i] + horizon:
            j += 1
        idx[i] = j if j < n else n

    return idx

def finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000)):

    ts = snapshots["ts"].to_numpy()
    mid = snapshots["mid"].to_numpy()
    n = len(snapshots)

    for h in horizons_ms:
        idx = forward_index(ts, h)

        future_mid = np.full(n, np.nan)
        valid = idx < n

        future_mid[valid] = mid[idx[valid]]
        
        # Store future mid
        snapshots[f"future_mid_{h}ms"] = future_mid

        # Store return
        snapshots[f"future_return_{h}ms"] = (future_mid - mid) / mid

    return snapshots

def compute_utility(snapshots, lambda_inv=1.0, gamma_tail=1.0):

    df = snapshots.copy()

    pnl = df["total_pnl"].values
    inventory = df["inventory"].values

    # 1. PnL flow (NOT level)
    pnl_flow = np.diff(pnl, prepend=pnl[0])

    pnl_term = np.mean(pnl_flow)

    # 2. Inventory risk (correct time-aligned)
    inv_risk = np.mean(inventory**2)

    # 3. Tail risk (on pnl flow, not level)
    losses = -pnl_flow

    var = np.quantile(losses, 0.95)
    cvar = losses[losses >= var].mean()

    utility = pnl_term - lambda_inv * inv_risk - gamma_tail * cvar

    print("PnL term:", pnl_term)
    print("Inventory penalty:", lambda_inv * inv_risk)
    print("Tail penalty:", gamma_tail * cvar)
    print("Utility:", utility)

    return utility

def generate_datasets(snapshots, trades, quotes, fills, events):
    snapshots = snapshots.sort_values("ts").copy()
    trades = trades.sort_values("ts").copy()
    quotes = quotes.sort_values("ts").copy()
    fills = fills.sort_values("ts").copy()
    events = events.sort_values("ts").copy()

    fills = label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000))
    
    trades = add_trade_impact_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = label_toxicity_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = align_trades_to_snapshots(trades, snapshots)

    snapshots = finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000))

    compute_utility(snapshots)

    snapshots.to_parquet(os.path.join(folder_path, "snapshots.parquet"), index=False)
    trades.to_parquet(os.path.join(folder_path, "trades.parquet"), index=False)
    quotes.to_parquet(os.path.join(folder_path, "quotes.parquet"), index=False)
    fills.to_parquet(os.path.join(folder_path, "fills.parquet"), index=False)
    events.to_parquet(os.path.join(folder_path, "events.parquet"), index=False)

generate_datasets(snapshots, trades, quotes, fills, events)

PnL term: -0.0025648991779696887
Inventory penalty: 0.6036086695083988
Tail penalty: 0.05219107959372017
Utility: -0.6583646482800886
